# ***ChatBot***

### Importing Libraries

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

True

### MODEL SETUP
*gpt-5-nano: cheapest capable OpenAI model as of April 2026*

*$0.05/1M input, $0.40/1M output*

*Temperature=0.7 => slight creativity for coaching*

In [2]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

### CHAT CHAIN
*System prompt: specific persona = consistent, useful behavior*

In [3]:
SYSTEM_PROMPT = """You are a senior backend engineering interview coach with 10+ years 
of experience at FAANG companies. You help engineers prepare for technical interviews.

Your behavior:
- Ask ONE technical question at a time from these areas: system design, DSA, 
  databases, APIs, distributed systems, or backend concepts
- After the candidate answers, give brutally honest feedback:
  * What they got RIGHT (be specific)
  * What they MISSED or got WRONG (be specific)  
  * A BETTER version of their answer (show them what a strong answer looks like)
- Then ask your next question

ALWAYS:
- Be direct and specific — no vague praise
- Point out exact gaps in knowledge
- Use real terminology (ACID, CAP theorem, O(n) complexity, etc.)

Start by asking what area they want to practice, then begin questioning."""

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chat_chain = chat_prompt | llm | StrOutputParser()

### SUMMARY CHAIN
*Separate chain for summarizing conversation every 5 turns*

*Uses lower temperature, summaries should be factual, not creative*

In [5]:
summary_llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

summary_prompt = ChatPromptTemplate.from_template(
    """
Summarize this interview coaching session in 3-5 bullet points.
Focus on: topics covered, candidate's strengths, gaps identified, and areas to study.

Conversation:
{conversation}

Summary:"""
)

summary_chain = summary_prompt | summary_llm | StrOutputParser()

### CONVERSATION STATE

In [6]:
chat_history: list[ HumanMessage | AIMessage] = []
turn_count = 0

### CORE CHAT FUNCTION AND OTHER FUNCTIONS

In [12]:
def chat(input: str) -> str:
    """
    Run one turn of the coaching conversation.
    Injects full history into prompt so the LLM has full context.
    Appends both messages to history after each turn.
    """
    global turn_count

    response = chat_chain.invoke(
        {
            "input": input, 
            "history": chat_history,
        }
    )

    chat_history.append(HumanMessage(content=input))
    chat_history.append(AIMessage(content=response))

    turn_count += 1

    return response

In [8]:
def get_conversation_summary() -> str:
    """
    Summarize the conversation so far using a separate LLM call.
    Called automatically every 5 turns.
    """
    if not chat_history:
        return "No conversation yet."
    
    conversation = "\n".join([
        f"{'Candidate' if isinstance(msg, HumanMessage) else 'Coach'}: {msg.content}"
        for msg in chat_history
    ])
    return summary_chain.invoke({"conversation": conversation})

In [9]:
def print_stats():
    """Print session statistics when user quits."""

    total_messages = len(chat_history)
    human_messages = sum(1 for m in chat_history if isinstance(m, HumanMessage))
    ai_messages = sum(1 for m in chat_history if isinstance(m, AIMessage))

    print("\n" + "="*60)
    print("\nSession Summary:")
    print("="*60)
    print(f"Turn count: {turn_count}")
    print(f"Total messages: {total_messages}")
    print(f"Human messages: {human_messages}")
    print(f"AI messages: {ai_messages}")

    if chat_history:
        print("\nFinal Conversation Summary:")
        print("-"*60)
        print(get_conversation_summary())
    print("="*60 + "\n")
    

### MAIN LOOP

In [10]:
def main():
    print("="*60)
    print("Hello, World! I am your ChatBot AI.")
    print(f"Model: {llm.model}, Temperature: {llm.temperature}")
    print("Type 'exit' to quit and see session stats.")
    print("="*60)
    print()

    opening = chat("Hello, I want to practice for backend engineering interviews. Please be BRUTALLY honest with me and give me specific feedback after each question.")
    print(f"Coach: {opening}\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print_stats()
            print("\nExiting...")
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit", "q"}:
            print_stats()
            print("\nExiting...")
            break

        response = chat(user_input)
        print(f"Coach: {response}\n")

        if turn_count > 0 and turn_count % 5 == 0:
            print("Generating conversation summary...")
            print("-"*60)
            print(get_conversation_summary())
            print("-"*60 + "\n")
            print()
        

## *RUN THE CHATBOT*

In [13]:
if __name__ == "__main__":
    main()

Hello, World! I am your ChatBot AI.
Model: gpt-5-nano, Temperature: None
Type 'exit' to quit and see session stats.

Coach: Which area would you like to practice? Choose one:

- System design
- Data structures & algorithms (DSA)
- Databases
- APIs
- Distributed systems
- Backend concepts

Pick one and I’ll ask a single, focused question in that area.

Coach: API question:

Design a RESTful API for a bookmarking service that is used by individual users (multi-tenant). You should provide:

- Endpoints (example: POST /bookmarks, GET /bookmarks/{id}, GET /bookmarks with pagination, filtering by tag, and a search query, PUT /bookmarks/{id}, DELETE /bookmarks/{id}).
- Authentication and authorization: Bearer token (e.g., JWT) must prove the user; users can only access their own bookmarks.
- Data model: Bookmark with at least id, user_id, url, title, description, tags (array), created_at, updated_at.
- ID generation: server-generated IDs (not client-supplied).
- Performance and indexing: how 